# **Rider Segmentation in Action**

## **Business Understandning**

Some explanation

## **Data Understanding**

In [ ]:
# Import necessary libraries
import os
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Set display options for pandas and seaborn
pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')

In [ ]:
# Define data and output directories
DATA_DIR = './data/divvy_tripdata/'
OUTPUT_DIR = './data/output/'
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
# Get list of all CSV files for 2023, 2024, and 2025
pattern = os.path.join(DATA_DIR, '2023*-divvy-tripdata.csv')
files_2023 = sorted(glob.glob(pattern))
pattern = os.path.join(DATA_DIR, '2024*-divvy-tripdata.csv')
files_2024 = sorted(glob.glob(pattern))
pattern = os.path.join(DATA_DIR, '2025*-divvy-tripdata.csv')
files_2025 = sorted(glob.glob(pattern))

# Combine all files into a single list
all_files = files_2023 + files_2024 + files_2025

# Print the number of files and a sample of the file names
len(all_files), all_files[:5], all_files[-5:]

In [ ]:
# Load and concatenate all CSV files into a single DataFrame
frames = []
for file in all_files:
    month_tag = os.path.basename(file).split('-')[0]  # Extract month from filename
    df = pd.read_csv(file)
    df["file_month"] = month_tag  # Add a column to identify the source file
    frames.append(df)

# Concatenate all DataFrames
all_df = pd.concat(frames, ignore_index=True)
print(f"Combined DataFrame shape: {all_df.shape}")
print(all_df.head())

## **Data Preparation**

In [ ]:
# Check for missing columns in the combined DataFrame
expected_cols = [
    'ride_id', 'rideable_type', 'started_at', 'ended_at', 'start_station_name', 'end_station_name',
    'start_lat', 'start_lng', 'end_lat', 'end_lng', 'member_casual', 'file_month'
]

# Verify that all expected columns are present in the combined DataFrame
missing_cols = [col for col in expected_cols if col not in all_df.columns]

if missing_cols:
    print(f"Warning: Missing columns in the combined DataFrame: {missing_cols}")
else:
    print("All expected columns are present in the combined DataFrame.")

In [ ]:
# Parse timestamps and calculate trip duration
all_df['started_at'] = pd.to_datetime(all_df['started_at'])
all_df['ended_at'] = pd.to_datetime(all_df['ended_at'])
all_df['ride_length_min'] = (all_df['ended_at'] - all_df['started_at']).dt.total_seconds() / 60  # Duration in minutes
all_df['ride_length_min'] = all_df['ride_length_min'].clip(lower=0)  # Ensure no negative durations
print(all_df[['started_at', 'ended_at', 'ride_length_min']].head())

# Clean and standardize categorical columns
all_df['rideable_type'] = all_df['rideable_type'].astype(str).str.strip().str.lower().str.replace(' ', '_')
all_df['member_casual'] = all_df['member_casual'].astype(str).str.strip().str.lower().str.replace(' ', '_')

# Filter out any rows with unexpected values in 'member_casual' or 'rideable_type'
all_df = all_df[all_df['member_casual'].isin(['member', 'casual'])]
all_df = all_df[all_df['rideable_type'].isin(['classic_bike', 'electric_bike', 'docked_bike'])]

# Final check of the cleaned DataFrame
all_df.shape, all_df['rideable_type'].unique(), all_df['member_casual'].unique()


## **Modeling**

## **Evaluation**

## **Deployment**